# Shapes of Stories: Curated Demo

Canonical books processed through GPT-2 medium.
Overlapping chunks to eliminate boundary artifacts.
Passage-level heuristics to label prose modes.
Exports JSON for interactive visualisation.

In [ ]:
!pip install -q transformers datasets umap-learn scikit-learn matplotlib pandas scipy 2>/dev/null

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import json, gc, os, warnings, random, re
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE = '/content/story_demo'
os.makedirs(SAVE, exist_ok=True)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
print(f'Device: {device}')

## 1. Curated Corpus

Recognisable books from Project Gutenberg. We search by title
and take a 10,000-token segment from the middle of each.

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Books we want — title substrings to match against pg19
WANTED = [
    'Pride and Prejudice',
    'Frankenstein',
    'Dracula',
    'A Christmas Carol',
    'Adventures of Sherlock Holmes',
    'Heart of Darkness',
    'War of the Worlds',
    'Alice\'s Adventures in Wonderland',
    'Moby Dick',
    'Picture of Dorian Gray',
    'Jane Eyre',
    'Wuthering Heights',
    'Great Expectations',
    'Tale of Two Cities',
    'The Odyssey',
    'Crime and Punishment',
    'Anna Karenina',
    'Treasure Island',
    'The Jungle Book',
    'The Time Machine',
    'The Strange Case of Dr. Jekyll',
    'The Importance of Being Earnest',
    'The Turn of the Screw',
    'Sense and Sensibility',
    'Little Women',
    'The Scarlet Letter',
    'The Call of the Wild',
    'Twenty Thousand Leagues',
    'Les Misérables',
    'Don Quixote',
]

SEG_LEN = 10000
MIN_BOOK_LEN = 15000

wanted_lower = [w.lower() for w in WANTED]
found = {}  # title -> book data

print(f'Searching pg19 for {len(WANTED)} books...')
pg = load_dataset('emozilla/pg19', split='train', streaming=True)

for ex in pg:
    title = ex.get('short_book_title', ex.get('title', ''))
    title_lower = title.lower()
    
    for wi, wanted in enumerate(wanted_lower):
        if wanted in title_lower and WANTED[wi] not in found:
            text = ex['text']
            toks = tokenizer.encode(text)
            if len(toks) < MIN_BOOK_LEN:
                continue
            
            mid = len(toks) // 2
            start = mid - SEG_LEN // 2
            seg_toks = toks[start:start + SEG_LEN]
            
            found[WANTED[wi]] = {
                'title': title,
                'search_key': WANTED[wi],
                'tokens': seg_toks,
                'full_len': len(toks),
                'seg_text': tokenizer.decode(seg_toks)
            }
            print(f'  Found: {title} ({len(toks)} tokens)')
            break
    
    if len(found) >= len(WANTED):
        break

# Also grab some extras to fill out if we didn't find all
if len(found) < 15:
    print(f'\nOnly found {len(found)}, grabbing additional books...')
    extra_count = 0
    pg2 = load_dataset('emozilla/pg19', split='train', streaming=True)
    for ex in pg2:
        title = ex.get('short_book_title', ex.get('title', ''))
        if title in [v['title'] for v in found.values()]:
            continue
        text = ex['text']
        toks = tokenizer.encode(text)
        if len(toks) < MIN_BOOK_LEN:
            continue
        mid = len(toks) // 2
        seg_toks = toks[mid - SEG_LEN//2 : mid + SEG_LEN//2]
        found[f'extra_{extra_count}'] = {
            'title': title, 'search_key': title,
            'tokens': seg_toks, 'full_len': len(toks),
            'seg_text': tokenizer.decode(seg_toks)
        }
        extra_count += 1
        print(f'  Extra: {title}')
        if len(found) >= 20:
            break

books = list(found.values())
print(f'\n{len(books)} books ready')
for b in books:
    print(f'  {b["title"]} ({b["full_len"]} tokens)')

## 2. Collect Hidden States (Overlapping Chunks)

Process in overlapping 1024-token chunks with 512 overlap.
Keep only hidden states from the second half of each chunk
(where the model has full context). This eliminates the
chunk boundary artifact.

In [ ]:
from transformers import GPT2LMHeadModel

MODEL_NAME = 'gpt2-medium'
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device); model.eval()
config = model.config
D_MODEL = config.n_embd
N_LAYERS = config.n_layer
CTX = config.n_positions
LAYER = N_LAYERS - 1  # penultimate
OVERLAP = 512  # overlap between chunks
STRIDE = CTX - OVERLAP  # 512 new tokens per chunk

print(f'Model: {MODEL_NAME}, d={D_MODEL}, layers={N_LAYERS}')
print(f'Chunks: {CTX} tokens, stride {STRIDE}, overlap {OVERLAP}')
print(f'Collecting layer {LAYER}')

In [ ]:
all_hidden = []   # per book: (n_tokens, D_MODEL)
all_entropy = []  # per book: (n_tokens,)
all_tokens = []   # per book: list of token ids (for text reconstruction)

print(f'Processing {len(books)} books with overlapping chunks...')
with torch.no_grad():
    for bi, book in enumerate(books):
        toks = book['tokens']
        book_h = []
        book_ent = []
        book_tok_ids = []
        
        chunk_starts = list(range(0, len(toks) - CTX + 1, STRIDE))
        if not chunk_starts:
            chunk_starts = [0]
        
        for ci, cs in enumerate(chunk_starts):
            chunk = toks[cs:cs+CTX]
            ids = torch.tensor([chunk], device=device)
            out = model(ids, output_hidden_states=True)
            
            h = out.hidden_states[LAYER][0].cpu()
            lp = F.log_softmax(out.logits[0], dim=-1)
            ent = -(torch.exp(lp) * lp).sum(dim=-1).cpu().numpy()
            
            if ci == 0:
                # First chunk: keep everything
                book_h.append(h)
                book_ent.append(ent)
                book_tok_ids.extend(chunk)
            else:
                # Subsequent chunks: keep only the non-overlapping part
                book_h.append(h[OVERLAP:])
                book_ent.append(ent[OVERLAP:])
                book_tok_ids.extend(chunk[OVERLAP:])
            
            del out, lp
        
        all_hidden.append(torch.cat(book_h, dim=0))
        all_entropy.append(np.concatenate(book_ent))
        all_tokens.append(book_tok_ids)
        print(f'  {bi+1}. {book["title"][:40]}: {all_hidden[-1].shape[0]} tokens')

del model; torch.cuda.empty_cache(); gc.collect()
print(f'\nDone. {sum(h.shape[0] for h in all_hidden)} total tokens.')

## 3. Window Pooling + Passage Features

W=64 tokens per passage. For each passage, compute:
- Mean hidden state (768-d)
- Text snippet
- Prose mode heuristics: quote density, mean sentence length, entropy

In [ ]:
WINDOW = 64

passage_vecs = []
passage_meta = []
story_ranges = []

for bi, h in enumerate(all_hidden):
    n_windows = h.shape[0] // WINDOW
    if n_windows < 4: continue
    
    start = len(passage_vecs)
    for wi in range(n_windows):
        s_tok = wi * WINDOW
        e_tok = (wi + 1) * WINDOW
        
        # Hidden state
        passage_vecs.append(h[s_tok:e_tok].mean(dim=0).numpy())
        
        # Decode passage text
        tok_ids = all_tokens[bi][s_tok:e_tok]
        passage_text = tokenizer.decode(tok_ids)
        
        # Prose mode heuristics
        n_quotes = passage_text.count('"') + passage_text.count("'") + passage_text.count('``') + passage_text.count("''")
        quote_density = n_quotes / max(len(passage_text), 1)
        
        # Sentence length proxy: count periods, exclamations, question marks
        n_sentences = len(re.findall(r'[.!?]+', passage_text))
        mean_sent_len = len(passage_text.split()) / max(n_sentences, 1)
        
        # Entropy
        mean_ent = all_entropy[bi][s_tok:e_tok].mean()
        
        # Dialogue fraction: rough estimate from quote marks
        in_quotes = 0
        inside = False
        for ch in passage_text:
            if ch == '"':
                inside = not inside
            elif inside:
                in_quotes += 1
        dialogue_frac = in_quotes / max(len(passage_text), 1)
        
        passage_meta.append({
            'book': bi,
            'title': books[bi]['title'],
            'window': wi,
            'pos_frac': (wi + 0.5) / n_windows,
            'text_snippet': passage_text[:150].replace('\n', ' '),
            'full_text': passage_text.replace('\n', ' '),
            'quote_density': quote_density,
            'dialogue_frac': dialogue_frac,
            'mean_sent_len': mean_sent_len,
            'entropy': float(mean_ent)
        })
    
    story_ranges.append((start, len(passage_vecs)))

passage_vecs = np.stack(passage_vecs)
n_books = len(story_ranges)
print(f'{n_books} books, {len(passage_vecs)} passages')

## 4. Autoencoder

In [ ]:
from sklearn.decomposition import IncrementalPCA, PCA

# PCA pre-reduction
print('PCA 1024 → 128...')
pca_pre = IncrementalPCA(n_components=128, batch_size=2000)
for i in range(0, len(passage_vecs), 2000): pca_pre.partial_fit(passage_vecs[i:i+2000])
pv_reduced = np.empty((len(passage_vecs), 128), dtype=np.float32)
for i in range(0, len(passage_vecs), 2000):
    chunk = passage_vecs[i:i+2000]
    pv_reduced[i:i+len(chunk)] = pca_pre.transform(chunk)

ev = pca_pre.explained_variance_ratio_
cs = np.cumsum(ev)
print(f'95% at {int(np.searchsorted(cs, 0.95))+1}, 99% at {int(np.searchsorted(cs, 0.99))+1}')

In [ ]:
class AE(nn.Module):
    def __init__(s, ind, hid, lat):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(ind, hid), nn.ReLU(),
                              nn.Linear(hid, hid//2), nn.ReLU(),
                              nn.Linear(hid//2, lat))
        s.dec = nn.Sequential(nn.Linear(lat, hid//2), nn.ReLU(),
                              nn.Linear(hid//2, hid), nn.ReLU(),
                              nn.Linear(hid, ind))
    def encode(s, x): return s.enc(x)
    def forward(s, x): z = s.enc(x); return s.dec(z), z

def train_ae(data, k, ind=128, hid=256, ep=80, bs=256, lr=1e-3):
    data_t = torch.tensor(data, dtype=torch.float32)
    n = len(data_t); perm = torch.randperm(n); nt = int(0.85*n)
    tr, te = data_t[perm[:nt]], data_t[perm[nt:]]
    m = AE(ind, hid, k).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, ep)
    ld = DataLoader(TensorDataset(tr), batch_size=bs, shuffle=True)
    for e in range(ep):
        m.train()
        for (b,) in ld:
            b = b.to(device); r, z = m(b)
            l = F.mse_loss(r, b); opt.zero_grad(); l.backward(); opt.step()
        sch.step()
    m.eval()
    with torch.no_grad():
        rt, _ = m(tr[:1000].to(device)); ft = F.mse_loss(rt, tr[:1000].to(device)).item()
        re2, _ = m(te.to(device)); fte = F.mse_loss(re2, te.to(device)).item()
    return m, ft, fte

sweep = {}
for k in [4, 8, 16]:
    print(f'k={k}...', end=' ')
    m, ft, fte = train_ae(pv_reduced, k)
    r = fte/ft if ft > 1e-15 else float('inf')
    sweep[k] = {'ft': ft, 'fte': fte, 'model': m}
    print(f'train={ft:.3e} test={fte:.3e} ratio={r:.2f}x')

In [ ]:
K = 8  # ← adjust
ae = sweep[K]['model']; ae.eval()

with torch.no_grad():
    pv_t = torch.tensor(pv_reduced, dtype=torch.float32)
    latents = torch.cat([ae.encode(pv_t[i:i+500].to(device)).cpu()
                         for i in range(0, len(pv_t), 500)]).numpy()
print(f'Latent: {latents.shape}')

## 5. UMAP + Visualisations

In [ ]:
import umap

print('UMAP...')
umap_emb = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(latents)

# Store coordinates in metadata
for i, m in enumerate(passage_meta):
    m['umap_x'] = float(umap_emb[i, 0])
    m['umap_y'] = float(umap_emb[i, 1])
    m['latent'] = latents[i].tolist()

In [ ]:
# Colour palette for books
book_titles = [books[passage_meta[s]['book']]['title'] for s, e in story_ranges]
n_colors = len(story_ranges)
if n_colors <= 10:
    cmap = plt.cm.tab10
elif n_colors <= 20:
    cmap = plt.cm.tab20
else:
    cmap = plt.cm.gist_ncar
colors = [cmap(i / n_colors) for i in range(n_colors)]

In [ ]:
# Main trajectory plot
fig, ax = plt.subplots(figsize=(18, 16))
ax.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=1, alpha=0.08)

from matplotlib.lines import Line2D
legend_elements = []

for bi, (s, e) in enumerate(story_ranges):
    traj = umap_emb[s:e]
    if len(traj) < 2: continue
    title = books[passage_meta[s]['book']]['title']
    short_title = title[:35]
    
    ax.plot(traj[:,0], traj[:,1], '-', color=colors[bi], alpha=0.45, linewidth=0.8)
    sizes = np.linspace(6, 25, len(traj))
    ax.scatter(traj[:,0], traj[:,1], c=[colors[bi]], s=sizes, alpha=0.5, zorder=5)
    ax.scatter(traj[0,0], traj[0,1], c=[colors[bi]], s=70, marker='o',
               edgecolors='black', linewidths=1.2, zorder=10)
    ax.scatter(traj[-1,0], traj[-1,1], c=[colors[bi]], s=70, marker='s',
               edgecolors='black', linewidths=1.2, zorder=10)
    
    legend_elements.append(Line2D([0],[0], color=colors[bi], linewidth=2, label=short_title))

legend_elements += [
    Line2D([0],[0], marker='o', color='grey', markeredgecolor='black', markersize=8, linestyle='None', label='Start'),
    Line2D([0],[0], marker='s', color='grey', markeredgecolor='black', markersize=8, linestyle='None', label='End')
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper right', framealpha=0.9)
ax.set_title('Shapes of Stories: Book Trajectories Through Narrative Space', fontsize=15)
ax.set_aspect('equal')
plt.tight_layout(); plt.savefig(f'{SAVE}/trajectories_curated.png', dpi=200); plt.show()

In [ ]:
# Individual book panels — 4-5 per row
n_per_row = 5
n_rows = (n_books + n_per_row - 1) // n_per_row
fig, axes = plt.subplots(n_rows, n_per_row, figsize=(5*n_per_row, 5*n_rows))
axes = axes.flat if n_rows > 1 else [axes] if n_books == 1 else axes

for bi, (s, e) in enumerate(story_ranges):
    ax = axes[bi]
    ax.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=0.3, alpha=0.05)
    traj = umap_emb[s:e]
    ax.plot(traj[:,0], traj[:,1], '-', color=colors[bi], alpha=0.5, linewidth=0.8)
    ax.scatter(traj[:,0], traj[:,1], c=[colors[bi]], s=8, alpha=0.6)
    ax.scatter(traj[0,0], traj[0,1], c=[colors[bi]], s=50, marker='o', edgecolors='black', linewidths=1)
    ax.scatter(traj[-1,0], traj[-1,1], c=[colors[bi]], s=50, marker='s', edgecolors='black', linewidths=1)
    title = books[passage_meta[s]['book']]['title'][:30]
    ax.set_title(title, fontsize=9, color=colors[bi])
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

for ax in list(axes)[n_books:]: ax.set_visible(False)
plt.suptitle('Individual Book Trajectories', fontsize=14)
plt.tight_layout(); plt.savefig(f'{SAVE}/individual_trajectories.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Prose Mode Analysis

Test whether UMAP position correlates with prose mode heuristics.

In [ ]:
# Colour by prose features
dialogue_fracs = np.array([m['dialogue_frac'] for m in passage_meta])
entropies = np.array([m['entropy'] for m in passage_meta])
sent_lens = np.array([m['mean_sent_len'] for m in passage_meta])
pos_fracs = np.array([m['pos_frac'] for m in passage_meta])

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, data, cmap_name, title in zip(axes.flat, 
    [dialogue_fracs, entropies, sent_lens, pos_fracs],
    ['YlOrRd', 'plasma', 'viridis', 'coolwarm'],
    ['Dialogue Fraction', 'Model Entropy', 'Mean Sentence Length', 'Position in Book']):
    sc = ax.scatter(umap_emb[:,0], umap_emb[:,1], c=data, cmap=cmap_name, s=3, alpha=0.4)
    plt.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(title, fontsize=12); ax.set_aspect('equal')

plt.suptitle('What Does the Narrative Space Encode?', fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig(f'{SAVE}/prose_features.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Quantitative: correlation of UMAP coords with features
from scipy.stats import spearmanr

print('Spearman correlations with UMAP position:')
print(f'{"Feature":<25} {"UMAP-x":>10} {"UMAP-y":>10}')
print('-' * 45)
for name, data in [('Dialogue fraction', dialogue_fracs),
                    ('Entropy', entropies),
                    ('Sentence length', sent_lens),
                    ('Position in book', pos_fracs)]:
    rx, _ = spearmanr(umap_emb[:,0], data)
    ry, _ = spearmanr(umap_emb[:,1], data)
    print(f'{name:<25} {rx:>10.3f} {ry:>10.3f}')

# Also correlate with latent axes
print(f'\nSpearman correlations with AE latent axes:')
print(f'{"Feature":<25}', ' '.join([f'{"Ax"+str(i):>8}' for i in range(K)]))
print('-' * (25 + 9*K))
for name, data in [('Dialogue fraction', dialogue_fracs),
                    ('Entropy', entropies),
                    ('Sentence length', sent_lens),
                    ('Position in book', pos_fracs)]:
    corrs = [spearmanr(latents[:,ki], data)[0] for ki in range(K)]
    print(f'{name:<25}', ' '.join([f'{c:>8.3f}' for c in corrs]))

In [ ]:
# Test: within Tenterhooks-like books, does the oscillation track dialogue?
# Find the book with highest trajectory variance (most oscillation)

traj_variances = []
for bi, (s, e) in enumerate(story_ranges):
    traj = umap_emb[s:e]
    var = np.var(traj, axis=0).sum()
    title = books[passage_meta[s]['book']]['title']
    traj_variances.append((bi, var, title))

traj_variances.sort(key=lambda x: -x[1])
print('Most oscillating books:')
for bi, var, title in traj_variances[:5]:
    s, e = story_ranges[bi]
    df = dialogue_fracs[s:e]
    print(f'  {title[:40]}: traj_var={var:.1f}, dialogue_frac={df.mean():.3f}±{df.std():.3f}')

# For the most oscillating book, plot trajectory coloured by dialogue fraction
top_bi = traj_variances[0][0]
s, e = story_ranges[top_bi]
traj = umap_emb[s:e]
df_vals = dialogue_fracs[s:e]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))

a1.scatter(umap_emb[:,0], umap_emb[:,1], c='lightgrey', s=1, alpha=0.05)
a1.plot(traj[:,0], traj[:,1], '-', color='grey', alpha=0.3, linewidth=0.5)
sc = a1.scatter(traj[:,0], traj[:,1], c=df_vals, cmap='YlOrRd', s=30, alpha=0.8, zorder=5)
plt.colorbar(sc, ax=a1, shrink=0.7, label='Dialogue fraction')
a1.set_title(f'{traj_variances[0][2][:40]}\nTrajectory Coloured by Dialogue', fontsize=11)
a1.set_aspect('equal')

# Timeline: dialogue fraction and UMAP-x over passage index
x = np.linspace(0, 1, len(traj))
a2.plot(x, (traj[:,0] - traj[:,0].mean()) / traj[:,0].std(), 'b-', alpha=0.7, label='UMAP-x (normalised)')
a2.plot(x, (df_vals - df_vals.mean()) / max(df_vals.std(), 1e-8), 'r-', alpha=0.7, label='Dialogue frac (normalised)')
a2.set_xlabel('Position in book'); a2.set_ylabel('Normalised value')
a2.set_title('Does Trajectory Position Track Dialogue?', fontsize=11)
a2.legend(); a2.grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig(f'{SAVE}/dialogue_test.png', dpi=150); plt.show()

## 7. Mean Profiles + Archetypes

In [ ]:
from scipy.interpolate import interp1d
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_RESAMPLE = 50
all_resampled = []
shape_story_map = []

for si in range(n_books):
    s, e = story_ranges[si]
    z = latents[s:e]
    if len(z) < 4: continue
    x_orig = np.linspace(0, 1, len(z))
    x_new = np.linspace(0, 1, N_RESAMPLE)
    resampled = np.zeros((N_RESAMPLE, K))
    for ki in range(K):
        resampled[:, ki] = interp1d(x_orig, z[:, ki], kind='linear')(x_new)
    all_resampled.append(resampled)
    shape_story_map.append(si)

all_resampled = np.stack(all_resampled)
shape_story_map = np.array(shape_story_map)
shape_vecs = all_resampled.reshape(len(all_resampled), -1)

sil_scores = []
for nc in range(2, min(8, n_books)):
    km = KMeans(n_clusters=nc, random_state=42, n_init=10)
    cl = km.fit_predict(shape_vecs)
    sil = silhouette_score(shape_vecs, cl)
    sil_scores.append((nc, sil))
    print(f'k={nc}: silhouette={sil:.4f}')

best_nc = max(sil_scores, key=lambda x: x[1])[0]
km = KMeans(n_clusters=best_nc, random_state=42, n_init=10)
arch_labels = km.fit_predict(shape_vecs)

print(f'\nBest: {best_nc} archetypes')
for ai in range(best_nc):
    members = np.where(arch_labels == ai)[0]
    titles = [books[passage_meta[story_ranges[shape_story_map[m]][0]]['book']]['title'][:35] for m in members]
    print(f'  Type {ai+1}: {titles}')

## 8. Export JSON for Interactive Visualisation

In [ ]:
# Build export structure
export_data = {
    'metadata': {
        'model': MODEL_NAME,
        'layer': LAYER,
        'window': WINDOW,
        'ae_k': K,
        'n_books': n_books,
        'n_passages': len(passage_meta)
    },
    'books': []
}

for bi, (s, e) in enumerate(story_ranges):
    book_idx = passage_meta[s]['book']
    book_data = {
        'title': books[book_idx]['title'],
        'full_length': books[book_idx]['full_len'],
        'archetype': int(arch_labels[bi]) if bi < len(arch_labels) else -1,
        'passages': []
    }
    
    for pi in range(s, e):
        m = passage_meta[pi]
        book_data['passages'].append({
            'x': round(m['umap_x'], 4),
            'y': round(m['umap_y'], 4),
            'pos': round(m['pos_frac'], 3),
            'text': m['text_snippet'],
            'dialogue': round(m['dialogue_frac'], 3),
            'entropy': round(m['entropy'], 2),
            'sent_len': round(m['mean_sent_len'], 1)
        })
    
    export_data['books'].append(book_data)

export_path = f'{SAVE}/story_shapes.json'
with open(export_path, 'w') as f:
    json.dump(export_data, f)

size_mb = os.path.getsize(export_path) / 1e6
print(f'Exported to {export_path} ({size_mb:.1f} MB)')
print(f'{len(export_data["books"])} books, {sum(len(b["passages"]) for b in export_data["books"])} passages')

# Also save for download
from google.colab import files
files.download(export_path)

## 9. Summary

In [ ]:
print('='*60)
print('SHAPES OF STORIES — CURATED DEMO')
print('='*60)
print(f'Model: {MODEL_NAME} (layer {LAYER}/{N_LAYERS})')
print(f'Books: {n_books}')
for bi, (s, e) in enumerate(story_ranges):
    title = books[passage_meta[s]['book']]['title']
    n_pass = e - s
    print(f'  {title[:45]}: {n_pass} passages')
print(f'\nTotal passages: {len(passage_vecs)} (W={WINDOW})')
print(f'AE: PCA-128 → k={K}')
print(f'Overlapping chunks: stride={STRIDE}, overlap={OVERLAP}')
print(f'Archetypes: {best_nc}')
print(f'\nJSON exported to {export_path}')